# <center>PTA Gravitational Wave Searches - part 1</center>

### <center><span style="font-family:Comic Sans MS; color:blue">Prepared by Valentina Di Marco</span></center>

<div style = 'background-color:yellow'> Before running the tutorial make a directory using your name and save a copy of Common_signal_search.ipynb  in it </div>

In this tutorial, we are going to use ENTERPRISE (Enhanced Numerical Toolbox Enabling a Robust PulsaR Inference SuitE... achronym or a backronym?), a pulsar timing analysis code developed by NANOGrav. ENTERPRISE is used for noise analysis, gravitational-wave searches, and timing model analysis. We will analyse some simulated data containing both a gravitational wave and noise.

ENTERPRISE can be used to analyse single pulsar noise (estimate the parameter of the noise sorces).
It can also be used to do model selection using Savage-Dickey approximation to calculate Bayes factors.
See a NANOGrav tutorial here
https://colab.research.google.com/drive/1OQBCqWhFkQStD2tECPjkB2FvdOK3g-9o?usp=sharing#scrollTo=hQvw86KeGNIz

In this tutorial, we’ll analyze the gravitational wave background. Because the sampling process can be time-consuming, the tutorial is split into two parts. In this first part, we’ll focus on setting up and running the sampler. The second part will guide you through estimating the gravitational wave background parameters and computing the optimal statistic used for detection, using previously generated chains.

In [2]:
import os
import sys
import glob
import json
import numpy as np
import subprocess
from enterprise import constants as const
from enterprise.pulsar import Pulsar
from enterprise.signals import signal_base
from enterprise.signals import white_signals
from enterprise.signals import gp_signals
from enterprise.signals import parameter
from enterprise.signals import selections
from enterprise.signals import gp_priors
from enterprise.signals import utils
from enterprise.signals import deterministic_signals
from enterprise.signals import gp_bases
from enterprise_extensions.blocks import common_red_noise_block
from enterprise_extensions import hypermodel
from enterprise_extensions.frequentist.optimal_statistic import OptimalStatistic as OS

from ppta_dr3_models import *
from ppta_dr3_utils import *

Optional mpi4py package is not installed.  MPI support is not available.
Optional acor package is not installed. Acor is optionally used to calculate the effective chain length for output in the chain file.


First we define some parameters and read in some data

The DE421 ephemeris contains estimates of the Moon’s and planets’ orbits. NASA’s Jet Propulsion Laboratory (JPL) periodically updates the ephemeris as more accurate measurements become available, with the latest version being DE441. In this tutorial, we use the older DE421 version to match the simulations I already have. This choice doesn't affect the purpose of the tutorial.

In [3]:
'''
Common Red Noise (CRN) SELECTIONS
'pl_nocorr_freegam'  :  #power law, no correlation, free spectral index
'pl_nocorr_fixgam'   :  #power law, no correlation, fixed 13/3 spectral index
'pl_hd_fixgam'       :  #power law red noise, fixed spectral index, Hellings-Downs correlations
'pl_hd_freegam'      :  #power law red noise, free spectral index, Hellings-Downs correlations
'''

# Define directories, ephemeris, the CRN selection, and chain number
datadir = '/fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array'
chainnum = '1' # number of the chain in case we want to run more than one chains
ephem = 'DE421'
CRN_Name = 'pl_nocorr_freegam' # which model for the CRN


# Define the list of pulsars in your array
psrnames = ['J0030+0451',
            'J0125-2327',
            'J0437-4715',
            'J0613-0200',
            'J0614-3329',
            'J0711-6830',
            'J0900-3144',
            'J1017-7156',
            'J1022+1001',
            'J1024-0719',
            'J1045-4509',
            'J1125-6014',
            'J1446-4701',
            'J1545-4550',
            'J1600-3053',
            'J1603-7202',
            'J1643-1224',
            'J1713+0747',
            'J1730-2304',
            'J1744-1134',
            'J1832-0836',
            'J1857+0943',
            'J1902-5105',
            'J1909-3744',
            'J1933-6211',
            'J1939+2134',
            'J2124-3358',
            'J2129-5721',
            'J2145-0750',
            'J2241-5236']

#read in par and tim files
parfiles = []
timfiles = []
suffix = ''
for psrname in psrnames:
    parfiles.append(datadir + '/' + psrname + suffix + '.par')
    timfiles.append(datadir + '/' + psrname + suffix + '.tim')

We are now going to load the pulsars into a class list

- The enterprise Pulsar class uses libstempo to load the .par and .tim files, then compiles all the relevant pulsar data into a single Pulsar object. This object holds everything needed for further pulsar and PTA analysis.

- When you run the next cell, you might see warnings like **WARNING: Could not find pulsar distance for PSR ....** Don’t worry—this is normal. Some pulsars don’t have well-measured distances, so they’re automatically assigned a default value of 1 Kpc with a 20% uncertainty.

In [4]:
psrs = construct_psrs(parfiles, timfiles, None, None, 
                      chainnum, ephem, min_mjd = None, max_mjd = None, load_existing = True)

tspan, fundamental_freq = get_tspan_fundamental_freq(psrs)
print('TSpan is {:.3f} days'.format(tspan / 86400.0))
print('Fundamental frequency is {:.3g} Hz'.format(fundamental_freq))

loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0030+0451.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0030+0451.tim
loading...[tempo2Util.C:396] Warning: [TIM1] Please place MODE flags in the parameter file 
 /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0125-2327.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0125-2327.tim


loading...[tempo2Util.C:401] Warning: [DUP1] duplicated warnings have been suppressed.
 /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0437-4715.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0437-4715.tim
loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0613-0200.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0613-0200.tim
loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0614-3329.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0614-3329.tim


loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0711-6830.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0711-6830.tim
loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0900-3144.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J0900-3144.tim


loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1017-7156.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1017-7156.tim


loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1022+1001.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1022+1001.tim
loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1024-0719.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1024-0719.tim
loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1045-4509.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1045-4509.tim
loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1125-6014.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1125-6014.tim


loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1446-4701.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1446-4701.tim


... (2 outputs left out)

loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1857+0943.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1857+0943.tim
loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1902-5105.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1902-5105.tim


loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1909-3744.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1909-3744.tim
loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1933-6211.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1933-6211.tim


loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1939+2134.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J1939+2134.tim
loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J2124-3358.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J2124-3358.tim
loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J2129-5721.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J2129-5721.tim
loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J2145-0750.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J2145-0750.tim
loading... /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J2241-5236.par /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/J2241-5236.tim


TSpan is 7975.010 days
Fundamental frequency is 1.45e-09 Hz


This next function creates a signal in which the timing-model parameters are marginalized out (integrated out) rather than explicitly sampled. Here <mark>use_svd=True</mark> indicates that the code uses a singular value decomposition (SVD) of the timing model’s design matrix to improve numerical stability and efficiency.

In [5]:
tm = gp_signals.MarginalizingTimingModel(use_svd=True)

Define the white noise model

In [7]:
"""
Define white noise model
"""
# EFAC
efac_prior = parameter.Uniform(0.0,10.0)
wn = white_signals.MeasurementNoise(
    efac=efac_prior,
    selection=low_freq)

wn += white_signals.MeasurementNoise(
   efac=efac_prior,
   selection=mid_freq)

wn += white_signals.MeasurementNoise(
   efac=efac_prior,
   selection=high_freq)

# EQUAD
log10_equad_prior = parameter.Uniform(-10,-4)
wn += white_signals.TNEquadNoise(
    log10_tnequad=log10_equad_prior,
    selection=low_freq)

wn += white_signals.TNEquadNoise(
   log10_tnequad=log10_equad_prior,
   selection=mid_freq)

wn += white_signals.TNEquadNoise(
    log10_tnequad=log10_equad_prior,
    selection=high_freq)

# ECORR
log10_ecorr_prior = parameter.Uniform(-10,-4)
wn += white_signals.EcorrKernelNoise(
    log10_ecorr=log10_ecorr_prior,
    selection=low_freq, name='basis_ecorr_low',
    method="sparse")
wn += white_signals.EcorrKernelNoise(
    log10_ecorr=log10_ecorr_prior,
    selection=mid_freq, name='basis_ecorr_mid',
    method="sparse")
wn += white_signals.EcorrKernelNoise(
    log10_ecorr=log10_ecorr_prior,
    selection=high_freq, name='basis_ecorr_high',
    method="sparse")

Set up the common noise

In [8]:
crn_model_dict = get_crn_model_dict(tspan)
print(CRN_Name)
crn = crn_model_dict[CRN_Name]
if len(CRN_Name[0]) > 1:
    for crn_name_ in CRN_Name[0][1:]:
        crn += crn_model_dict[crn_name_.replace('_quad_', '')]

pl_nocorr_freegam


Combine everything into a model "s" before adding the pulsar-specific noise

In [9]:
s = tm + wn + crn

Add the pulsar-specific noise

In [18]:
model = []
priors = {}
components_dict = {key: [] for key in ['red', 'dm', 'band', 'chrom', 'hf']}
for psr in psrs:
    """
    Define red noise model
    """
    rn_model, rn_lgA_prior, rn_gam_prior, priors = get_informed_rednoise_priors(psr, 'red_noise', {}, {}, 
                                    priors, use_basic_priors=True, log10_A_min_basic=-18)

    Tspan = psr.toas.max() - psr.toas.min()  # seconds
    max_cadence = 240.0  # days
    red_components = int(Tspan / (max_cadence*86400))
    components_dict['red'].append(red_components)
    print("Using {} red noise components".format(red_components))
    rn = gp_signals.FourierBasisGP(rn_model, components=red_components,
                                   selection=no_selection, name='red_noise')
    
    """
    Define DM noise model
    """

    dm_model, dm_lgA_prior, dm_gam_prior, priors = get_informed_rednoise_priors(psr, 'dm_gp', {}, {}, 
                                    priors, use_basic_priors=True, log10_A_min_basic=-18)
    Tspan = psr.toas.max() - psr.toas.min()  # seconds
    max_cadence = 60  # days
    dm_components = int(Tspan / (max_cadence*86400))
    components_dict['dm'].append(dm_components)
    print("Using {} DM components".format(dm_components))
    dm_basis = gp_bases.createfourierdesignmatrix_dm(nmodes=dm_components)
    dm = gp_signals.BasisGP(dm_model, dm_basis, name='dm_gp')
    
    """
    Define chromatic noise model
    """

    chrom_model, chrom_lgA_prior, chrom_gam_prior, priors = get_informed_rednoise_priors(psr, 'chrom_gp', {}, {}, 
                                    priors, use_basic_priors=True, log10_A_min_basic=-18)

    idx = 4  # Define freq^-idx scaling
    max_cadence = 240  # days
    chrom_components = int(Tspan / (max_cadence*86400))
    components_dict['chrom'].append(chrom_components)
    print("Using {} Chrom components".format(chrom_components))
    chrom_basis = gp_bases.createfourierdesignmatrix_chromatic(nmodes=chrom_components,
                                                               idx=idx)
    chrom = gp_signals.BasisGP(chrom_model, chrom_basis, name='chrom_gp')
    
    """
    define solar wind model
    """
    def get_informed_nearth_priors(psr, priors):
        key_ = psr.name + '_gp_sw_log10_A'
        print('getting prior for {}'.format(key_))
        log10_A_min = -10
        log10_A_max = -3
        gamma_min = -4
        gamma_max = 4
        log10_A_sw = parameter.Uniform(log10_A_min, log10_A_max)
        gamma_sw = parameter.Uniform(gamma_min, gamma_max)
        print(f"""{psr.name}_gp_sw_noise prior:
        log10_A in [{log10_A_min}, {log10_A_max}]
        gamma in [{gamma_min}, {gamma_max}]
        """)
        key_ = psr.name + '_gp_sw_log10_A'
       	priors[key_ + "_min"] = log10_A_min
       	priors[key_ + "_max"] = log10_A_max
       	key_ = psr.name + '_gp_sw_gamma'
       	priors[key_ + "_min"] = gamma_min
       	priors[key_ + "_max"] = gamma_max        

        Tspan = psr.toas.max() - psr.toas.min()
        max_cadence = 60
        sw_components = int(Tspan / (max_cadence*86400))
        sw_prior = utils.powerlaw(log10_A=log10_A_sw, gamma=gamma_sw)
        sw_basis = createfourierdesignmatrix_solar_dm(nmodes=sw_components, Tspan=Tspan)
        
        n_earth = parameter.Constant(4)
        deter_sw = solar_wind(n_earth=n_earth)
        mean_sw = deterministic_signals.Deterministic(deter_sw)
        sw = mean_sw
        sw += gp_signals.BasisGP(sw_prior, sw_basis, name='gp_sw')

        return sw, priors

    sw, priors_sw = get_informed_nearth_priors(psr, priors)
    
    """
    Define total model by summing all components
    """
    # define model, add DM variations
    s2 = s + rn  + dm + ch + sw
    model.append(s2(psr))

getting prior for J0030+0451_red_noise_log10_A
J0030+0451_red_noise_noise prior:
    log10_A in [-18, -11]
    gamma in [0, 7]
    
Using 33 red noise components
getting prior for J0030+0451_dm_gp_log10_A
J0030+0451_dm_gp_noise prior:
    log10_A in [-18, -11]
    gamma in [0, 7]
    
Using 132 DM components
getting prior for J0030+0451_chrom_gp_log10_A
J0030+0451_chrom_gp_noise prior:
    log10_A in [-18, -11]
    gamma in [0, 7]
    
Using 33 Chrom components
getting prior for J0030+0451_gp_sw_log10_A
J0030+0451_gp_sw_noise prior:
        log10_A in [-10, -3]
        gamma in [-4, 4]
        
getting prior for J0125-2327_red_noise_log10_A
J0125-2327_red_noise_noise prior:
    log10_A in [-18, -11]
    gamma in [0, 7]
    
Using 33 red noise components
getting prior for J0125-2327_dm_gp_log10_A
J0125-2327_dm_gp_noise prior:
    log10_A in [-18, -11]
    gamma in [0, 7]
    
Using 132 DM components
getting prior for J0125-2327_chrom_gp_log10_A
J0125-2327_chrom_gp_noise prior:
    log10

getting prior for J1909-3744_red_noise_log10_A
J1909-3744_red_noise_noise prior:
    log10_A in [-18, -11]
    gamma in [0, 7]
    
Using 33 red noise components
getting prior for J1909-3744_dm_gp_log10_A
J1909-3744_dm_gp_noise prior:
    log10_A in [-18, -11]
    gamma in [0, 7]
    
Using 132 DM components
getting prior for J1909-3744_chrom_gp_log10_A
J1909-3744_chrom_gp_noise prior:
    log10_A in [-18, -11]
    gamma in [0, 7]
    
Using 33 Chrom components
getting prior for J1909-3744_gp_sw_log10_A
J1909-3744_gp_sw_noise prior:
        log10_A in [-10, -3]
        gamma in [-4, 4]
        
getting prior for J1933-6211_red_noise_log10_A
J1933-6211_red_noise_noise prior:
    log10_A in [-18, -11]
    gamma in [0, 7]
    
Using 33 red noise components
getting prior for J1933-6211_dm_gp_log10_A
J1933-6211_dm_gp_noise prior:
    log10_A in [-18, -11]
    gamma in [0, 7]
    
Using 132 DM components
getting prior for J1933-6211_chrom_gp_log10_A
J1933-6211_chrom_gp_noise prior:
    log10

We can now set up the sampler

In [19]:
"""
Set up your PTA likelihood and your sampler
"""
# set up PTA
nmodels = 1
pta = dict.fromkeys(np.arange(nmodels))
pta[0] = signal_base.PTA(model)

hyper_model = hypermodel.HyperModel(pta)

#params = hyper_model.param_names
#print(params)
#chaindir = datadir + '/noiseFiles/chains/'
#print(chaindir + '*chain.npy')
##try:
#    chainfiles = sorted([sorted(glob.glob(chaindir + '{}*chain.npy'.format(psr_.name)))[0] for psr_ in psrs])
#except IndexError:
#    chainfiles = []
#print('Found {} chain files'.format(len(chainfiles)))

#print("")
#print("Priors:")
#print(priors)
#print("")

#setting up initial samples by sampling from single-pulsar noise runs: speed up burn-in
x0 = hyper_model.initial_sample()
ndim = len(x0)

# sampler for N steps
N = int(1e6)

savename = 'commonNoise' + '_' + ephem
outdir = datadir + '/chains/'  + '/' + savename + '/' + chainnum
print('saving to {}'.format(outdir))

# Use PTMCMC sampler. We can easily update this to use e.g. Bilby instead
sampler = hyper_model.setup_sampler(outdir=outdir, resume=False)

# Print parameter names and write to a file
filename = outdir + "/pars.txt"
if os.path.exists(filename):
    os.remove(filename)
with open(filename, "a") as f:
    for par in hyper_model.param_names:
        f.write(par + '\n')
filename = outdir + "/custom_priors.json"
with open(filename, 'w') as f:
    json.dump(priors, f)

# print components to file:
filename = outdir + "/components.json"
if os.path.exists(filename):
    os.remove(filename)
with open(filename, "w") as f:
    json.dump(components_dict, f)

# copy current file to output directory
os.system('cp {0} {1}/{0}'.format(os.path.basename(sys.argv[0]), outdir))

Duplicate signal J0030+0451_measurement_noise from objects <Enterprise Signal object measurement_noise[J0030+0451_mid_efac]> and <Enterprise Signal object measurement_noise[J0030+0451_efac]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J0030+0451_measurement_noise from objects <Enterprise Signal object measurement_noise[J0030+0451_high_efac]> and <Enterprise Signal object measurement_noise[J0030+0451_efac]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J0030+0451_tnequad from objects <Enterprise Signal object tnequad[J0030+0451_mid_log10_tnequad]> and <Enterprise Signal object tnequad[J0030+0451_log10_tnequad]>.
This functionality was ad

Duplicate signal J0711-6830_measurement_noise from objects <Enterprise Signal object measurement_noise[J0711-6830_high_efac]> and <Enterprise Signal object measurement_noise[J0711-6830_efac]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J0711-6830_tnequad from objects <Enterprise Signal object tnequad[J0711-6830_mid_log10_tnequad]> and <Enterprise Signal object tnequad[J0711-6830_log10_tnequad]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J0711-6830_tnequad from objects <Enterprise Signal object tnequad[J0711-6830_high_log10_tnequad]> and <Enterprise Signal object tnequad[J0711-6830_log10_tnequad]>.
This functionality was added in v1.1

Duplicate signal J1045-4509_tnequad from objects <Enterprise Signal object tnequad[J1045-4509_mid_log10_tnequad]> and <Enterprise Signal object tnequad[J1045-4509_log10_tnequad]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J1045-4509_tnequad from objects <Enterprise Signal object tnequad[J1045-4509_high_log10_tnequad]> and <Enterprise Signal object tnequad[J1045-4509_log10_tnequad]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J1125-6014_measurement_noise from objects <Enterprise Signal object measurement_noise[J1125-6014_mid_efac]> and <Enterprise Signal object measurement_noise[J1125-6014_efac]>.
This functionality was added in v1.1.

Duplicate signal J1603-7202_tnequad from objects <Enterprise Signal object tnequad[J1603-7202_high_log10_tnequad]> and <Enterprise Signal object tnequad[J1603-7202_log10_tnequad]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J1643-1224_measurement_noise from objects <Enterprise Signal object measurement_noise[J1643-1224_mid_efac]> and <Enterprise Signal object measurement_noise[J1643-1224_efac]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J1643-1224_measurement_noise from objects <Enterprise Signal object measurement_noise[J1643-1224_high_efac]> and <Enterprise Signal object measurement_noise[J1643-1224_efac]>.
This functionality was a

Duplicate signal J1857+0943_measurement_noise from objects <Enterprise Signal object measurement_noise[J1857+0943_mid_efac]> and <Enterprise Signal object measurement_noise[J1857+0943_efac]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J1857+0943_measurement_noise from objects <Enterprise Signal object measurement_noise[J1857+0943_high_efac]> and <Enterprise Signal object measurement_noise[J1857+0943_efac]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J1857+0943_tnequad from objects <Enterprise Signal object tnequad[J1857+0943_mid_log10_tnequad]> and <Enterprise Signal object tnequad[J1857+0943_log10_tnequad]>.
This functionality was ad

Duplicate signal J2124-3358_measurement_noise from objects <Enterprise Signal object measurement_noise[J2124-3358_high_efac]> and <Enterprise Signal object measurement_noise[J2124-3358_efac]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J2124-3358_tnequad from objects <Enterprise Signal object tnequad[J2124-3358_mid_log10_tnequad]> and <Enterprise Signal object tnequad[J2124-3358_log10_tnequad]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J2124-3358_tnequad from objects <Enterprise Signal object tnequad[J2124-3358_high_log10_tnequad]> and <Enterprise Signal object tnequad[J2124-3358_log10_tnequad]>.
This functionality was added in v1.1

saving to /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/data/array/chains//commonNoise_DE421/1
Adding red noise prior draws...

Adding DM GP noise prior draws...

Adding Solar Wind DM GP prior draws...

Adding GWB uniform distribution draws...

Adding gw param prior draws...

Adding nmodel uniform distribution draws...



cp: cannot stat 'ipykernel_launcher.py': No such file or directory


256

Finally we can sample. Running this cell takes a long time, so instead we are going to jump into the plots tutorial /fred/oz005/users/pulsar_workshop/valentina2/PTA_tutorials/Plots.ipynb and run that using existing chains. 

In [20]:
# Sample! The sampler parameters can be left as default.
sampler.sample(x0, N, SCAMweight=30, AMweight=15, DEweight=50)

Finished 0.10 percent in 110.802150 s Acceptance rate = 0.642

KeyboardInterrupt: 